# A/B Testing Framework for GPU Cloud Dashboard Optimization

## 1. Project Overview

This project analyzes GPU cloud workload data and builds a simulated A/B testing framework to evaluate whether a smart GPU dashboard can improve job completion outcomes.

The dataset includes job-level workload information such as GPU model, CPU request, GPU request, worker count, submit time, job duration, and job type. A separate node dataset provides infrastructure context, including GPU capacity and CPU capacity by node.

The goal is to demonstrate how A/B testing can be applied to AI infrastructure and developer platform optimization using Python.

## 2. Business Problem

GPU cloud users often submit jobs without full visibility into GPU availability, expected queue behavior, workload complexity, or failure risk. This can lead to failed, abandoned, or inefficient jobs.

The business problem is to evaluate whether a smarter GPU dashboard could improve job completion by giving users better guidance before and during job execution.

In this experiment:

- Variant A represents the standard dashboard.
- Variant B represents a smart dashboard with GPU availability guidance, job-risk warnings, and configuration recommendations.

The main success metric is job completion rate.


## 3. Dataset Overview


In [7]:
import pandas as pd
import numpy as np
import plotly.express as px


df_job = pd.read_csv('/Users/mohamedamr/Desktop/A:B/job_info_df.csv')
df_node = pd.read_csv('/Users/mohamedamr/Desktop/A:B/node_info_df.csv')

In [8]:
print("Job dataset shape:", df_job.shape)
print("Node dataset shape:", df_node.shape)

Job dataset shape: (466867, 9)
Node dataset shape: (4278, 4)


In [9]:
node_summary = df_node.groupby('gpu_model').agg(
    node_count=('node_name', 'count'),
    total_gpu_capacity=('gpu_capacity_num', 'sum'),
    avg_cpu_capacity=('cpu_num', 'mean')
).reset_index()

node_summary

,gpu_model,node_count,total_gpu_capacity,avg_cpu_capacity
0,A10,2494,2494,127.998396
1,A100-SXM4-80GB,432,3456,128.000000
2,A800-SXM4-80GB,22,176,128.000000
3,GPU-series-1,989,1558,191.935288
4,GPU-series-2,122,976,192.000000
5,H800,219,1752,192.000000


In [10]:
fig = px.bar(
    node_summary,
    x='gpu_model',
    y='total_gpu_capacity',
    title='Total GPU Capacity by GPU Model',
    labels={
        'gpu_model': 'GPU Model',
        'total_gpu_capacity': 'Total GPU Capacity'
    }
)

fig.show()

In [11]:
fig = px.bar(
    node_summary,
    x='gpu_model',
    y='node_count',
    title='Node Count by GPU Model',
    labels={
        'gpu_model': 'GPU Model',
        'node_count': 'Number of Nodes'
    }
)

fig.show()

In [12]:
job_gpu_summary = df_job['gpu_model'].value_counts().reset_index()
job_gpu_summary.columns = ['gpu_model', 'job_count']

fig = px.bar(
    job_gpu_summary,
    x='gpu_model',
    y='job_count',
    title='Job Count by GPU Model',
    labels={
        'gpu_model': 'GPU Model',
        'job_count': 'Number of Jobs'
    }
)

fig.show()

The dataset overview shows both workload demand and infrastructure capacity. The job dataset shows that most submitted jobs request A10 and A100-SXM4-80GB GPUs, while the node dataset shows how many nodes and total GPU capacity are available for each GPU model. This context supports the business problem: users may benefit from a smarter dashboard that helps them understand resource availability and job risk before submitting workloads.

## 4. Data Cleaning and Preparation

In [13]:
df = df_job.copy()

df['duration_minutes'] = df['duration'] / 60
df['duration_hours'] = df['duration'] / 3600
df['submit_time_hours'] = df['submit_time'] / 3600


Because some jobs lasted thousands of hours, and those extreme values could distort our analysis.

In [14]:
duration_99 = df['duration_hours'].quantile(0.99)

df_clean = df[df['duration_hours'] <= duration_99].copy()
df_clean = df_clean.reset_index(drop=True)

print("Original rows:", df.shape[0])
print("Cleaned rows:", df_clean.shape[0])
print("Rows removed:", df.shape[0] - df_clean.shape[0])

Original rows: 466867
Cleaned rows: 462198
Rows removed: 4669


#  5. A/B Test Design

In [15]:
np.random.seed(42)

df_clean['variant'] = np.random.choice(
    ['A', 'B'],
    size=len(df_clean),
    p=[0.5, 0.5]
)

df_clean['variant'].value_counts()

variant
B    231333
A    230865
Name: count, dtype: int64

In [16]:
df_clean.groupby('variant')[[
    'cpu_request',
    'gpu_request',
    'worker_num',
    'duration_hours'
]].mean()

,cpu_request,gpu_request,worker_num,duration_hours
variant,,,,
A,24.443432,2.034188,1.685223,26.894000
B,24.376222,2.030790,1.710054,26.929438


* Randomly assigned jobs into Variant A and Variant B to simulate an A/B testing environment for scheduler performance evaluation.

* Defined job completion probabilities based on operational factors such as duration, GPU demand, worker count, and job type, while applying a positive treatment effect to Variant B.

* Simulated realistic job completion outcomes using a binomial distribution to compare performance differences between control and experimental groups.

The simulation behaves like:

About 72 out of 100 jobs become 1     but not guaranteed

About 28 out of 100 jobs become 0     but not guaranteed

#  6. Experiment Simulation

In [17]:
df_clean['completion_probability'] = 0.72

df_clean.loc[df_clean['duration_hours'] > 24, 'completion_probability'] -= 0.12
df_clean.loc[df_clean['gpu_request'] > 4, 'completion_probability'] -= 0.08
df_clean.loc[df_clean['worker_num'] > 10, 'completion_probability'] -= 0.06
df_clean.loc[df_clean['job_type'] == 'Spot', 'completion_probability'] -= 0.05

df_clean.loc[df_clean['variant'] == 'B', 'completion_probability'] += 0.04

df_clean['completion_probability'] = df_clean['completion_probability'].clip(0.05, 0.95)

df_clean.groupby('variant')['completion_probability'].mean()

variant
A    0.694416
B    0.734390
Name: completion_probability, dtype: float64

In [18]:
np.random.seed(123)

df_clean['random_number'] = np.random.random(size=len(df_clean))

df_clean['job_completed'] = (
    df_clean['random_number'] < df_clean['completion_probability']
).astype(int)

df_clean.groupby('variant')[['completion_probability', 'job_completed']].mean()

,completion_probability,job_completed
variant,,
A,0.694416,0.694358
B,0.734390,0.732926


* Performance improvement relative to baseline.  : 5.55% 

In [19]:
conversion_rates = df_clean.groupby('variant')['job_completed'].mean()

a_rate = conversion_rates['A']
b_rate = conversion_rates['B']

absolute_lift = b_rate - a_rate
relative_lift = (absolute_lift / a_rate) * 100

print(f"A completion rate: {a_rate:.4f}")
print(f"B completion rate: {b_rate:.4f}")
print(f"Absolute lift: {absolute_lift:.4f}")
print(f"Relative lift: {relative_lift:.2f}%")

A completion rate: 0.6944
B completion rate: 0.7329
Absolute lift: 0.0386
Relative lift: 5.55%


# 7. Statistical Testing


* The p-value is far below 0.05, so the difference between A and B is statistically significant.

* The negative z-score is okay because the test compares A - B. Since B is higher than A, the statistic becomes negative.

In [20]:
from statsmodels.stats.proportion import proportions_ztest

successes = [
    df_clean[df_clean['variant'] == 'A']['job_completed'].sum(),
    df_clean[df_clean['variant'] == 'B']['job_completed'].sum()
]

samples = [
    df_clean[df_clean['variant'] == 'A'].shape[0],
    df_clean[df_clean['variant'] == 'B'].shape[0]
]

z_stat, p_value = proportions_ztest(
    count=successes,
    nobs=samples
)

print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.6f}")

Z-statistic: -29.0018
P-value: 0.000000


We are 95% confident that the smart dashboard improves job completion by 3.60 to 4.12 percentage points.

In [21]:
from statsmodels.stats.proportion import confint_proportions_2indep

ci_low, ci_high = confint_proportions_2indep(
    count1=successes[1],
    nobs1=samples[1],
    count2=successes[0],
    nobs2=samples[0],
    method='wald'
)

print(f"95% confidence interval for lift: [{ci_low:.4f}, {ci_high:.4f}]")

95% confidence interval for lift: [0.0360, 0.0412]


# 8. Results and Visualization

In [22]:
results_summary_formatted = pd.DataFrame({
    'Metric': [
        'A Completion Rate',
        'B Completion Rate',
        'Absolute Lift',
        'Relative Lift',
        'P-value',
        '95% CI Lower',
        '95% CI Upper'
    ],
    'Value': [
        f"{a_rate:.2%}",
        f"{b_rate:.2%}",
        f"{absolute_lift:.2%}",
        f"{relative_lift:.2f}%",
        f"{p_value:.6f}",
        f"{ci_low:.2%}",
        f"{ci_high:.2%}"
    ]
})

results_summary_formatted

,Metric,Value
0,A Completion Rate,69.44%
1,B Completion Rate,73.29%
2,Absolute Lift,3.86%
3,Relative Lift,5.55%
4,P-value,0.000000
5,95% CI Lower,3.60%
6,95% CI Upper,4.12%


A: about 69.44%
B: about 73.29%
B is visibly higher than A

In [23]:
import plotly.express as px

plot_data = df_clean.groupby('variant')['job_completed'].mean().reset_index()
plot_data['completion_rate_percent'] = plot_data['job_completed'] * 100

fig = px.bar(
    plot_data,
    x='variant',
    y='completion_rate_percent',
    text=plot_data['completion_rate_percent'].round(2),
    title='Job Completion Rate by Dashboard Variant',
    labels={
        'variant': 'Dashboard Variant',
        'completion_rate_percent': 'Completion Rate (%)'
    }
)

fig.update_traces(texttemplate='%{text}%', textposition='outside')

fig.update_layout(
    yaxis_range=[0, 100],
    width=700,
    height=450
)

fig.show()

In [24]:
lift_data = pd.DataFrame({
    'Metric': ['Absolute Lift'],
    'Value': [absolute_lift * 100]
})

fig = px.bar(
    lift_data,
    x='Metric',
    y='Value',
    text=lift_data['Value'].round(2),
    title='Absolute Lift from Smart Dashboard Variant',
    labels={
        'Metric': '',
        'Value': 'Lift in Completion Rate (%)'
    }
)

fig.update_traces(
    texttemplate='%{text}%',
    textposition='outside'
)

fig.update_layout(
    yaxis_range=[0, 5],
    width=700,
    height=450
)

fig.show()

In [25]:
ci_plot_data = pd.DataFrame({
    'Metric': ['Estimated Lift'],
    'Lift': [absolute_lift * 100],
    'CI Lower': [ci_low * 100],
    'CI Upper': [ci_high * 100]
})

fig = px.scatter(
    ci_plot_data,
    x='Metric',
    y='Lift',
    error_y=[(ci_high - absolute_lift) * 100],
    error_y_minus=[(absolute_lift - ci_low) * 100],
    title='Estimated Lift with 95% Confidence Interval',
    labels={
        'Lift': 'Lift in Completion Rate (%)',
        'Metric': ''
    }
)

fig.update_traces(marker=dict(size=14))

fig.update_layout(
    yaxis_title='Lift (%)',
    width=700,
    height=450
)

fig.show()

# 9. Business Recommendation

The simulated A/B test shows that the smart GPU dashboard, Variant B, outperformed the standard dashboard, Variant A.

Variant A achieved a job completion rate of 69.44%, while Variant B achieved a completion rate of 73.29%. This represents an absolute lift of 3.86 percentage points and a relative lift of 5.55%.

A two-proportion z-test produced a p-value below 0.05, indicating that the observed difference is statistically significant. The 95% confidence interval for the lift ranges from 3.60% to 4.12%, meaning the estimated improvement is consistently positive.

Based on these results, Variant B should be recommended for rollout. The smart dashboard features, such as GPU availability guidance, job-risk warnings, and configuration recommendations, are expected to improve GPU job completion outcomes.

# 10. Limitations

This project uses real GPU workload and node capacity data, but the A/B assignment and job completion outcome are simulated because the original dataset does not include live experiment labels or completion-success records.

The purpose of the project is to demonstrate experiment design, statistical testing, and product analytics methodology for GPU cloud dashboard optimization.

# 11. Conclusion

This project demonstrates how A/B testing can be applied to GPU cloud product analytics. Using real GPU workload and node capacity data, the analysis simulated an experiment comparing a standard dashboard against a smart dashboard with better workload guidance.

The smart dashboard variant achieved a higher job completion rate, produced a statistically significant improvement, and showed a positive confidence interval for lift. Based on the simulated experiment, Variant B would be recommended for rollout.

This framework can be extended to real production experiments using actual user assignment, job completion logs, queue times, failure reasons, and user behavior data.